In [18]:
from training_utilities_2nd_part import *
from training_utilities import *

In [19]:
twitter_df = pd.read_csv("/Users/forough/PycharmProjects/mitigation3/Aiops_data_splitting_paper/code/Multivariate_time_series/twitter2/twitter3.csv")

twitter_df = twitter_df[4: len(twitter_df)-120]

twitter_df['Date'] = pd.to_datetime(twitter_df['Date'])
twitter_df['YearMonth'] = twitter_df['Date'].dt.to_period('M')
one_hot = pd.get_dummies(twitter_df['Symbol'], prefix='Symbol')
twitter_df = pd.concat([twitter_df, one_hot], axis=1)
twitter_df = twitter_df.drop(columns = ['Series', 'Trades', 'Symbol'])
twitter_df

,Date,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Deliverable Volume,%Deliverble,YearMonth,Symbol_ADANIPORTS,Symbol_MUNDRAPORT
0,2007-11-27,440.00,770.00,1050.00,770.00,959.0,962.90,984.72,27294366,2.687719e+15,9859619,0.3612,2007-11,False,True
1,2007-11-28,962.90,984.00,990.00,874.00,885.0,893.90,941.38,4581338,4.312765e+14,1453278,0.3172,2007-11,False,True
2,2007-11-29,893.90,909.00,914.75,841.00,887.0,884.20,888.09,5124121,4.550658e+14,1069678,0.2088,2007-11,False,True
3,2007-11-30,884.20,890.00,958.00,890.00,929.0,921.55,929.17,4609762,4.283257e+14,1260913,0.2735,2007-11,False,True
4,2007-12-03,921.55,939.75,995.00,922.00,980.0,969.30,965.65,2977470,2.875200e+14,816123,0.2741,2007-12,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3317,2021-04-26,725.35,733.00,739.65,728.90,729.2,730.75,733.25,9390549,6.885658e+14,838079,0.0892,2021-04,True,False
3318,2021-04-27,730.75,735.00,757.50,727.35,748.6,749.15,747.67,20573107,1.538191e+15,1779639,0.0865,2021-04,True,False
3319,2021-04-28,749.15,755.00,760.00,741.10,743.4,746.25,751.02,11156977,8.379106e+14,1342353,0.1203,2021-04,True,False
3320,2021-04-29,746.25,753.20,765.85,743.40,746.4,746.75,753.06,13851910,1.043139e+15,1304895,0.0942,2021-04,True,False


In [20]:
# twitter
# from variables_to_specify_twitter import *
columns_to_normalize = ['Prev Close', 'Open', 'High', 'Low', 'Last','Close', 'VWAP', 'Volume', 'Turnover', 'Deliverable Volume','%Deliverble']
twitter_target_col = 'Close'
forecast_avg_target_col_name = 'forecast_avg_Close'
avg_target_col_name = 'avg_Close'
No_of_datapoints_in_one_day = 1
date_col_name = 'Date'
one_month_window_size = 31
one_month_days =31
out_columns = ['Training dataset', 'Testing dataset', 'mae', 'mse', 'rmse', 'r2', 'mape', 'training_time',
              'Testing Error', 'testing_time']

twitter_drop_columnss = [twitter_target_col]+['YearMonth']+[date_col_name]
twitter_windows = [50,150,310,450,600,750,900]

print(twitter_target_col)

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
twitter_df[columns_to_normalize] = scaler.fit_transform(twitter_df[columns_to_normalize])
twitter_df = twitter_df.dropna().reset_index(drop=True)

twitter_time_steps = 1

daily_df_avg = twitter_df[['Close']].copy()
daily_df_avg.rename(columns={'Close': 'avg_Close'}, inplace=True)

Close


,Date,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Deliverable Volume,%Deliverble,YearMonth,Symbol_ADANIPORTS,Symbol_MUNDRAPORT
0,2007-11-27,0.276794,0.550634,0.774216,0.570576,0.709167,0.712743,0.734103,0.279227,0.329318,0.439703,0.322305,2007-11,False,True
1,2007-11-28,0.712743,0.728634,0.724774,0.659896,0.647500,0.655217,0.697799,0.046763,0.052818,0.064606,0.274102,2007-11,False,True
2,2007-11-29,0.655217,0.666251,0.662766,0.631554,0.649167,0.647130,0.653161,0.052318,0.055733,0.047490,0.155346,2007-11,False,True
3,2007-11-30,0.647130,0.650447,0.698406,0.673638,0.684167,0.678269,0.687572,0.047054,0.052456,0.056023,0.226227,2007-11,False,True
4,2007-12-03,0.678269,0.691828,0.728895,0.701121,0.726667,0.718079,0.718129,0.030347,0.035202,0.036176,0.226884,2007-12,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3317,2021-04-26,0.514694,0.519859,0.518479,0.535277,0.517667,0.519196,0.523459,0.095984,0.084346,0.037155,0.024321,2021-04,True,False
3318,2021-04-27,0.519196,0.521522,0.533188,0.533946,0.533833,0.534537,0.535537,0.210436,0.188457,0.079169,0.021363,2021-04,True,False
3319,2021-04-28,0.534537,0.538158,0.535248,0.545755,0.529500,0.532119,0.538344,0.114063,0.102646,0.059657,0.058392,2021-04,True,False
3320,2021-04-29,0.532119,0.536660,0.540068,0.547730,0.532000,0.532536,0.540052,0.141645,0.127794,0.057985,0.029798,2021-04,True,False


# stationary

In [21]:
twitter_len_of_training_data_of_stationary_model =258

train = twitter_df[0:twitter_len_of_training_data_of_stationary_model] 
test = twitter_df[twitter_len_of_training_data_of_stationary_model:]

eval_df_first_month, stationary_model1 = twitter_stationary_model_with_hptuning(train, test, 1, out_columns, twitter_target_col, twitter_drop_columnss)
sum_training_time_stat1 = eval_df_first_month['training_time'].sum()
print('sum_training_time is: ', sum_training_time_stat1)
print(eval_df_first_month['Testing Error'].mean())
print("mae is : ", eval_df_first_month['mae'].mean())

Model Type: RandomForestRegressor
Storage Required: 2.25 MB
model storage is:  2.245224952697754


total_time is:  0.1801577090000137
sum_training_time is:  0.1445272500000101
0.0030819595603613394
mae is :  0.033461880837028515


# Model reuse

In [22]:
seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 258)
seasonality_periods_acf = seasonality_periods_acf_ls[0]

Detected seasonality periods (ACF): [258 262 327 330 355 394 421 469 473 479 491]
median_value is:  394


# drift detection

In [23]:
df_copy = twitter_df[[twitter_target_col]]
target_col = twitter_target_col
time_steps = twitter_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = twitter_time_steps
x = 258* multiplier
window_len_=[x]
drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=twitter_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=51, Test size=51
Fold 1: Train size=102, Test size=51
Fold 2: Train size=153, Test size=51
Fold 3: Train size=204, Test size=51
Skipping fold 4: Insufficient training or test data.


In [24]:
eval_df_monthly2, avg_ml_storage1 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model1, twitter_len_of_training_data_of_stationary_model,twitter_df, "SA", twitter_target_col, twitter_drop_columnss, twitter_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)


window is:  258
i/window is :  1.0
Model Type: RandomForestRegressor
Storage Required: 2.25 MB


window is:  516
i/window is :  2.0
similar_month_index is :  0
month_index:  2




window is:  774
i/window is :  3.0
similar_month_index is :  1
month_index:  3




window is:  1032
i/window is :  4.0
Model Type: RandomForestRegressor
Storage Required: 2.09 MB


window is:  1290
i/window is :  5.0
similar_month_index is :  3
previous_model_i is :  1290
math.floor(previous_model_i/window) is:  5
len(models_ls) is: 4
Model Type: RandomForestRegressor
Storage Required: 2.07 MB


window is:  1548
i/window is :  6.0
Model Type: RandomForestRegressor
Storage Required: 2.07 MB


window is:  1806
i/window is :  7.0
Model Type: RandomForestRegressor
Storage Required: 2.21 MB


window is:  2064
i/window is :  8.0
similar_month_index is :  6
month_index:  8




window is:  2322
i/window is :  9.0
similar_month_index is :  7
month_index:  9




window is:  2580
i/window is :  10.0
Model Type: RandomFo

In [25]:
eval_df_monthly2, avg_ml_storage2 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model1, twitter_len_of_training_data_of_stationary_model,twitter_df, "SA", twitter_target_col, twitter_drop_columnss, twitter_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  258
i/window is :  1.0
Model Type: RandomForestRegressor
Storage Required: 2.25 MB


window is:  516
i/window is :  2.0
similar_month_index is :  0
month_index:  2




window is:  774
i/window is :  3.0
similar_month_index is :  1
month_index:  3




window is:  1032
i/window is :  4.0
similar_month_index is :  0
month_index:  4




window is:  1290
i/window is :  5.0
similar_month_index is :  1
month_index:  5




window is:  1548
i/window is :  6.0
similar_month_index is :  1
month_index:  6




window is:  1806
i/window is :  7.0
similar_month_index is :  0
month_index:  7




window is:  2064
i/window is :  8.0
similar_month_index is :  6
previous_model_i is :  2064
math.floor(previous_model_i/window) is:  8
len(models_ls) is: 7
Model Type: RandomForestRegressor
Storage Required: 2.18 MB


window is:  2322
i/window is :  9.0
Model Type: RandomForestRegressor
Storage Required: 2.18 MB


window is:  2580
i/window is :  10.0
Model Type: RandomForestRegressor
Storage Requir

In [26]:
eval_df_monthly2, avg_ml_storage3 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model1, twitter_len_of_training_data_of_stationary_model,twitter_df, "ES", twitter_target_col, twitter_drop_columnss, twitter_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  258
Model Type: RandomForestRegressor
Storage Required: 2.25 MB


window is:  516
Model Type: RandomForestRegressor
Storage Required: 2.22 MB


window is:  774
similar_month_index is :  0
month_index:  2



window is:  1032
similar_month_index is :  1
month_index:  3




window is:  1290
Model Type: RandomForestRegressor
Storage Required: 2.09 MB


window is:  1548
similar_month_index is :  3
previous_model_i is :  1548
math.floor(previous_model_i/window) is:  6
len(models_ls) is: 5
Model Type: RandomForestRegressor
Storage Required: 2.21 MB


window is:  1806
Model Type: RandomForestRegressor
Storage Required: 2.21 MB


window is:  2064
similar_month_index is :  1
month_index:  7




window is:  2322
similar_month_index is :  1
month_index:  8




window is:  2580
similar_month_index is :  7
previous_model_i is :  2580
math.floor(previous_model_i/window) is:  10
len(models_ls) is: 9
Model Type: RandomForestRegressor
Storage Required: 2.19 MB


window is:  2838
Model Type: 

In [27]:
eval_df_monthly2, avg_ml_storage4 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model1, twitter_len_of_training_data_of_stationary_model,twitter_df, "ES", twitter_target_col, twitter_drop_columnss, twitter_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  258
Model Type: RandomForestRegressor
Storage Required: 2.25 MB


window is:  516
Model Type: RandomForestRegressor
Storage Required: 2.22 MB


window is:  774
Model Type: RandomForestRegressor
Storage Required: 2.20 MB


window is:  1032
Model Type: RandomForestRegressor
Storage Required: 2.09 MB


window is:  1290
similar_month_index is :  2
month_index:  4




window is:  1548
similar_month_index is :  1
month_index:  5




window is:  1806
Model Type: RandomForestRegressor
Storage Required: 2.21 MB


window is:  2064
similar_month_index is :  4
previous_model_i is :  2064
math.floor(previous_model_i/window) is:  8
len(models_ls) is: 7
Model Type: RandomForestRegressor
Storage Required: 2.18 MB


window is:  2322
similar_month_index is :  2
month_index:  8




window is:  2580
similar_month_index is :  4
previous_model_i is :  2580
math.floor(previous_model_i/window) is:  10
len(models_ls) is: 9
Model Type: RandomForestRegressor
Storage Required: 2.19 MB


window is:  28

In [28]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print(avg_ml_storage_reuse)


2.180474403926304


# informed

In [29]:
informed_update(stationary_model1,twitter_df, target_col, twitter_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices, 1)

window is:  258
Model Type: RandomForestRegressor
Storage Required: 2.25 MB
window is:  516
window is:  774
window is:  1032
window is:  1290
Model Type: RandomForestRegressor
Storage Required: 2.09 MB
window is:  1548
window is:  1806
window is:  2064
Model Type: RandomForestRegressor
Storage Required: 2.18 MB
window is:  2322
window is:  2580
Model Type: RandomForestRegressor
Storage Required: 2.18 MB
window is:  2838
window is:  3096
Informed total_time:  0.7091758750000281
avg_ml_storage:  2.172818183898926


# periodical

In [30]:
periodical_retraining_with_hptuning(1, twitter_df, twitter_windows, out_columns, twitter_target_col, twitter_drop_columnss)

window is : 50
window size is :  50
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.46 MB
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.46 MB
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.46 MB
Model Type: RandomForestRegressor
Storage Required: 0.46 MB
Model Type: RandomForestRegressor
Storage Required: 0.45 MB
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.47 MB
Model Type: RandomForestRegressor
Storage Required: 0.46 MB
Model Type: RandomForestRegressor
Storage Required: 0.46 MB
Mode

([         Training dataset     Testing dataset       mae       mse      rmse  \
  0   trained on window i-1  tested on window i  0.125196  0.019767  0.140597   
  1   trained on window i-1  tested on window i  0.022102  0.001302  0.036087   
  2   trained on window i-1  tested on window i  0.018377  0.000422  0.020540   
  3   trained on window i-1  tested on window i  0.081777  0.009034  0.095047   
  4   trained on window i-1  tested on window i  0.012416  0.000196  0.014016   
  ..                    ...                 ...       ...       ...       ...   
  60  trained on window i-1  tested on window i  0.010002  0.000154  0.012419   
  61  trained on window i-1  tested on window i  0.003765  0.000025  0.005009   
  62  trained on window i-1  tested on window i  0.001677  0.000005  0.002248   
  63  trained on window i-1  tested on window i  0.060551  0.005553  0.074518   
  64  trained on window i-1  tested on window i  0.101285  0.014811  0.121700   
  
            r2       mape